# 03 - Regresion Polinomial y Funciones Base

---

## Contenido

1. Limitaciones de la Regresion Lineal
2. Funciones Base: la idea general
3. Funciones Base Polinomiales
4. PolynomialFeatures en Scikit-Learn
5. Pipelines de Scikit-Learn
6. Sobreajuste y el Dilema Sesgo-Varianza
7. Ejemplo con el dataset Iris

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set()

## 1. Limitaciones de la Regresion Lineal

Tal pareciera que la Regresion Lineal esta limitada **estrictamente** a modelar relaciones lineales... pero esto **no es cierto**. Podemos relajar esta condicion.

Para utilizar RL cuando existen relaciones no lineales entre las variables, podemos **transformar** las caracteristicas a traves de **funciones base**.

---

## 2. Funciones Base: la idea general

La idea es tomar nuestra Regresion Lineal Multiple:

$$y = a_0 + a_1 x_1 + a_2 x_2 + a_3 x_3 + \cdots$$

y **construir** las variables $x_1, x_2, x_3, \ldots$ como **funciones** de una variable original. Es decir, en lugar de usar directamente la variable $x$, se aplican transformaciones $\phi_j(x)$ denominadas funciones base:

$$y = a_0 + a_1 \phi_1(x) + a_2 \phi_2(x) + a_3 \phi_3(x) + \cdots$$

De esta forma, el modelo sigue siendo **lineal en los parametros** $a_j$, pero puede capturar **relaciones no lineales** en la variable original $x$.

Tipos comunes de funciones base:

| Tipo | Funciones base | Ejemplo |
|---|---|---|
| **Polinomial** | $\phi_j(x) = x^j$ | $y = a_0 + a_1 x + a_2 x^2 + a_3 x^3$ |
| **Gaussiana** | $\phi_j(x) = e^{-(x - \mu_j)^2 / 2\sigma^2}$ | Funciones radiales |
| **Sinusoidal** | $\phi_j(x) = \sin(jx)$ | Series de Fourier |

---

## 3. Funciones Base Polinomiales

El caso mas comun es el polinomial, donde las funciones base son potencias de $x$:

$$y = a_0 + a_1 x + a_2 x^2 + a_3 x^3 + \cdots + a_d x^d$$

Esto se conoce como **Regresion Polinomial**. La clave conceptual es que, aunque el modelo parece no lineal en $x$, sigue siendo **lineal en los parametros** $a_j$.

Lo que hacemos es proyectar una variable en 1D a un espacio de $d$ dimensiones, donde presumiblemente existe un hiperplano que podemos obtener mediante regresion lineal.

---

## 4. PolynomialFeatures en Scikit-Learn

Scikit-Learn provee el transformador `PolynomialFeatures` para generar estas proyecciones automaticamente:

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Ejemplo: transformar un vector de 1D a polinomios de grado 3
poly = PolynomialFeatures(degree=3, include_bias=False)

x_example = np.array([2, 3, 4])
x_poly = poly.fit_transform(x_example[:, np.newaxis])

print('Datos originales:', x_example)
print()
print('Datos transformados (grado 3):')
print(x_poly)
print()
print('Columnas: [x, x^2, x^3]')

Podemos observar que el arreglo `x` ha pasado de ser de 1D a 3D, al elevar cada valor por un exponente de acuerdo a su posicion.

Esto nos permite representar la misma informacion en un espacio de mayor dimensionalidad, en el cual presumiblemente existe un hiperplano que podemos obtener mediante regresion lineal.

---

## 5. Pipelines de Scikit-Learn

Para simplificar el encadenamiento de procesos, Scikit-Learn provee la clase `Pipeline`. Este objeto permite encadenar varios procesos. Por ejemplo:

```
crear variables polinomiales --> hacer una Regresion Lineal
```

La forma mas *limpia* de lograr esto es usando `make_pipeline`:

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression

# Crear un pipeline: PolynomialFeatures(grado 7) + LinearRegression
poly_model = make_pipeline(
    PolynomialFeatures(7),
    LinearRegression()
)

print('Pipeline creado:')
print(poly_model)
print()
print('Nota: El pipeline es independiente de los datos. Solo define la secuencia de pasos.')

### Ejemplo: Ajustar una onda senoidal con ruido

Usando esta transformacion en conjunto con una regresion lineal, podemos ajustar un conjunto de datos con una relacion mucho mas compleja entre $x$ e $y$:

In [ ]:
# Generar datos: onda senoidal con ruido
rng = np.random.RandomState(1)
x = 10 * rng.rand(50)
y = np.sin(x) + 0.1 * rng.randn(50)

# Ajustar el pipeline
poly_model.fit(x[:, np.newaxis], y)

# Predecir sobre un rango continuo
xfit = np.linspace(-1, 11, 1000)
yfit = poly_model.predict(xfit[:, np.newaxis])

# Visualizar
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(x, y, color='steelblue', alpha=0.7, label='Datos con ruido')
ax.plot(xfit, yfit, color='red', linewidth=2, label='Ajuste polinomial (grado 7)')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Regresion Polinomial: Ajuste a una onda senoidal', fontsize=13)
ax.set_ylim(-1.5, 1.5)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

Empleando funciones base polinomiales de 7mo orden, el ajuste obtenido por la funcion al conjunto de datos es muy bueno. Pero surge una pregunta importante: **que pasa si elegimos un grado demasiado alto o demasiado bajo?**

---

## 6. Sobreajuste y el Dilema Sesgo-Varianza

Las funciones base le dan mayor flexibilidad a nuestra regresion lineal, sin embargo, se corre el riesgo de **sobreentrenar** el modelo. Esto tambien se conoce como el **Dilema de Sesgo y Varianza**:

| Situacion | Grado bajo (subajuste) | Grado adecuado | Grado alto (sobreajuste) |
|---|---|---|---|
| **Sesgo** | Alto | Bajo | Bajo |
| **Varianza** | Baja | Baja | Alta |
| **Error total** | Alto | Bajo | Alto |
| **Descripcion** | El modelo es demasiado simple | Balance optimo | El modelo memoriza el ruido |

Veamos esto graficamente:

In [ ]:
# Comparar grado 2 (subajuste) vs grado 7 (adecuado) vs grado 20 (sobreajuste)
rng = np.random.RandomState(1)
x = 10 * rng.rand(50)
y = np.sin(x) + 0.1 * rng.randn(50)

degrees = [2, 7, 20]
titles = ['Subajuste (grado 2)', 'Ajuste adecuado (grado 7)', 'Sobreajuste (grado 20)']
colors = ['orange', 'green', 'purple']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, degree, title, color in zip(axes, degrees, titles, colors):
    model_temp = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model_temp.fit(x[:, np.newaxis], y)
    
    xfit = np.linspace(0, 10, 1000)
    yfit = model_temp.predict(xfit[:, np.newaxis])
    
    ax.scatter(x, y, color='steelblue', alpha=0.6, s=30)
    ax.plot(xfit, yfit, color=color, linewidth=2)
    ax.set_title(title, fontsize=12)
    ax.set_ylim(-2, 2)
    ax.set_xlabel('x')
    ax.set_ylabel('y')

plt.tight_layout()
plt.show()

Observaciones:

- **Grado 2 (subajuste):** El modelo es demasiado simple, asume que los datos se ajustan a una parabola.
- **Grado 7 (adecuado):** El modelo captura la tendencia general sin memorizar el ruido.
- **Grado 20 (sobreajuste):** El modelo tiene demasiada flexibilidad y llega a valores extremos, memorizando el ruido.

---

## 7. Ejemplo con el dataset Iris

Apliquemos la regresion polinomial al dataset Iris. Usaremos `sepal_length` para predecir `petal_length` y compararemos diferentes grados polinomiales.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# Cargar datos
iris = sns.load_dataset('iris')

X_iris = iris[['sepal_length']].values
y_iris = iris['petal_length'].values

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.30, random_state=42
)

print(f'Entrenamiento: {X_train.shape[0]} muestras')
print(f'Prueba: {X_test.shape[0]} muestras')

In [ ]:
# Comparar diferentes grados polinomiales
degrees = [1, 2, 3, 5]
results = []

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

x_range = np.linspace(X_iris.min() - 0.5, X_iris.max() + 0.5, 200).reshape(-1, 1)

for idx, degree in enumerate(degrees):
    # Crear y ajustar pipeline
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X_train, y_train)
    
    # Predecir
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    y_curve = model.predict(x_range)
    
    # Metricas
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
    
    results.append({
        'Grado': degree,
        'R2_train': r2_train,
        'R2_test': r2_test,
        'RMSE_test': rmse_test
    })
    
    # Graficar
    ax = axes[idx]
    ax.scatter(X_train, y_train, color='steelblue', alpha=0.5, label='Train', s=30)
    ax.scatter(X_test, y_test, color='orange', alpha=0.5, label='Test', s=30)
    ax.plot(x_range, y_curve, color='red', linewidth=2)
    ax.set_title(f'Grado {degree} | R2_test={r2_test:.3f}', fontsize=12)
    ax.set_xlabel('sepal_length')
    ax.set_ylabel('petal_length')
    ax.legend(fontsize=9)
    ax.set_ylim(0, 8)

plt.suptitle('Regresion Polinomial sobre Iris: sepal_length vs petal_length',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Tabla resumen de resultados
df_results = pd.DataFrame(results)
print('Comparacion de modelos polinomiales:')
print(df_results.to_string(index=False))

---

## Resumen

En este notebook aprendimos:

1. Las **funciones base** permiten extender la regresion lineal a relaciones no lineales.
2. Las **funciones base polinomiales** son las mas comunes: $\phi_j(x) = x^j$.
3. `PolynomialFeatures` de Scikit-Learn transforma automaticamente las variables.
4. Los **Pipelines** encadenan transformaciones y modelos en un solo objeto.
5. El **Dilema Sesgo-Varianza** advierte sobre el riesgo de subajuste y sobreajuste.
6. La eleccion del grado polinomial debe balancear la complejidad del modelo con su capacidad de generalizacion.

En el siguiente notebook veremos la **Seleccion de Variables con Correlacion y Mapas de Calor**.